# EVE Optimizer Benchmark: Tabular Data on MLP

**EVE (ρ=0.58, K=4)** vs **SGD**, **AdamW**, **RMSProp** on sklearn benchmark tabular datasets.

### Focus
- **Time to converge** (wall-clock seconds to target accuracy)
- **Optimizer metrics**: steps/sec, time per epoch, gradient norm trends
- **Results**: final accuracy, best validation accuracy

### Datasets
- Wine, Breast Cancer, Digits (small, fast for Colab)

### Setup
Run first cell to clone repo and install. Works on Colab with CPU or GPU.

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────
# Colab: clone repo. Local: pip install -e . from EVE directory, or run from repo root
import os, sys, subprocess
if os.path.exists("/content"):  # Colab
    subprocess.run(["git", "clone", "https://github.com/SattamAltwaim/EVE.git", "/content/EVE"],
                   capture_output=True, check=False)
    if "/content/EVE" not in sys.path:
        sys.path.insert(0, "/content/EVE")
else:
    # Local: add parent dir if running from benchmarks/
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.exists(os.path.join(parent, "eve_optimizer")) and parent not in sys.path:
        sys.path.insert(0, parent)

from eve_optimizer import EVE
print("EVE imported successfully")

In [ ]:
# ── 1. Imports & Config ──────────────────────────────────────────────────
import math
import time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_wine, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from eve_optimizer import EVE

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# EVE fixed params: rho=0.58, K=4
EVE_RHO, EVE_K = 0.58, 4

In [ ]:
# ── 2. Data loading ──────────────────────────────────────────────────────
def get_tabular_data(name: str):
    """Return (X_train, X_val, y_train, y_val, n_features, n_classes)."""
    if name == "wine":
        X, y = load_wine(return_X_y=True)
    elif name == "breast_cancer":
        X, y = load_breast_cancer(return_X_y=True)
    elif name == "digits":
        X, y = load_digits(return_X_y=True)
    else:
        raise ValueError(f"Unknown: {name}")

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    n_features = X_train.shape[1]
    n_classes = len(np.unique(y))
    return X_train, X_val, y_train, y_val, n_features, n_classes

def make_loaders(name: str, batch_size: int = 32):
    X_tr, X_val, y_tr, y_val, n_feat, n_cls = get_tabular_data(name)
    train_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.long)
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    return train_loader, val_loader, n_feat, n_cls

# Quick test
_, _, nf, nc = make_loaders("wine")
print(f"Wine: n_features={nf}, n_classes={nc}")

In [ ]:
# ── 3. MLP model ──────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, n_features: int, n_classes: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(), nn.BatchNorm1d(hidden),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.BatchNorm1d(hidden),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# ── 4. Optimizer factory & training ───────────────────────────────────────
@dataclass
class EpochMetrics:
    train_loss: float
    val_acc: float
    epoch_time: float
    steps_per_sec: float
    grad_norm: Optional[float]
    epoch: int

def make_optimizer(name: str, model: nn.Module, lr: float = 1e-3, wd: float = 0.01):
    if name == "SGD":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    elif name == "AdamW":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    elif name == "RMSProp":
        return torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)
    elif name == "EVE":
        return EVE(model.parameters(), lr=lr, weight_decay=wd, K=EVE_K, rho=EVE_RHO)
    raise ValueError(f"Unknown: {name}")

def grad_norm(model: nn.Module) -> float:
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return math.sqrt(total)

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

def train_one_epoch(model, loader, optimizer, device, opt_name, log_grad=False):
    model.train()
    total_loss, n = 0.0, 0
    grad_norms = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        if log_grad:
            grad_norms.append(grad_norm(model))
        if opt_name == "EVE":
            optimizer.step(model=model, loss_fn=F.cross_entropy, data=(x, y))
        else:
            optimizer.step()
        total_loss += loss.item() * y.size(0)
        n += y.size(0)
    return total_loss / n, np.mean(grad_norms) if grad_norms else None

In [ ]:
# ── 5. Benchmark runner ───────────────────────────────────────────────────
DATASETS = ["wine", "breast_cancer", "digits"]
OPTS = ["SGD", "AdamW", "RMSProp", "EVE"]
EPOCHS = 80
LR_MAP = {"SGD": 0.1, "AdamW": 1e-3, "RMSProp": 1e-3, "EVE": 1e-3}
TARGET_ACC = {"wine": 0.95, "breast_cancer": 0.95, "digits": 0.95}

def run_benchmark(seed: int = 42):
    results = defaultdict(dict)
    for ds_name in DATASETS:
        print(f"\n{'='*60}\n  {ds_name.upper()}\n{'='*60}")
        train_loader, val_loader, n_feat, n_cls = make_loaders(ds_name)
        target = TARGET_ACC[ds_name]

        for opt_name in OPTS:
            torch.manual_seed(seed)
            model = MLP(n_feat, n_cls).to(DEVICE)
            optimizer = make_optimizer(opt_name, model, LR_MAP[opt_name])

            history = []
            cum_time = 0.0
            time_to_target = None
            for ep in range(1, EPOCHS + 1):
                t0 = time.perf_counter()
                tr_loss, gn = train_one_epoch(
                    model, train_loader, optimizer, DEVICE, opt_name, log_grad=(ep == 1)
                )
                elapsed = time.perf_counter() - t0
                cum_time += elapsed
                val_acc = evaluate(model, val_loader, DEVICE)
                n_batches = len(train_loader)
                sps = n_batches / elapsed if elapsed > 0 else 0

                history.append(EpochMetrics(
                    train_loss=tr_loss, val_acc=val_acc,
                    epoch_time=elapsed, steps_per_sec=sps,
                    grad_norm=gn, epoch=ep
                ))
                if time_to_target is None and val_acc >= target:
                    time_to_target = cum_time
                if ep % 20 == 0 or ep == 1:
                    print(f"  {opt_name:8} ep={ep:3d} loss={tr_loss:.4f} val_acc={val_acc:.4f} "
                          f"time={elapsed:.2f}s steps/s={sps:.0f}")

            results[ds_name][opt_name] = {
                "history": history,
                "time_to_target": time_to_target,
                "best_val_acc": max(h.val_acc for h in history),
                "total_time": cum_time,
            }
            del model, optimizer
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
    return results

In [ ]:
# ── 6. Run benchmark ──────────────────────────────────────────────────────
results = run_benchmark()

In [ ]:
# ── 7. Optimizer metrics summary (focus: time to converge) ─────────────────
print("=" * 80)
print("  OPTIMIZER METRICS — Time to Converge & Efficiency")
print("=" * 80)
for ds in DATASETS:
    target = TARGET_ACC[ds]
    print(f"\n{ds.upper()} (target val_acc >= {target})")
    print("-" * 60)
    print(f"{'Optimizer':<10} {'Time to target (s)':<20} {'Best val_acc':<14} {'Total time (s)':<14} {'Avg s/epoch':<12}")
    for opt in OPTS:
        r = results[ds][opt]
        tt = r["time_to_target"]
        tt_str = f"{tt:.2f}" if tt is not None else "N/A"
        avg_ep = r["total_time"] / EPOCHS
        print(f"{opt:<10} {tt_str:<20} {r['best_val_acc']:.4f}          {r['total_time']:.2f}          {avg_ep:.3f}")

In [ ]:
# ── 8. Plots: Loss curves, Val accuracy & Time to target ───────────────────
colors = {"SGD": "#1f77b4", "AdamW": "#2ca02c", "RMSProp": "#ff7f0e", "EVE": "#d62728"}

# Training loss curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, ds in zip(axes, DATASETS):
    for opt in OPTS:
        hist = results[ds][opt]["history"]
        ax.plot([h.epoch for h in hist], [h.train_loss for h in hist],
                label=opt, color=colors[opt])
    ax.set_title(ds)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train Loss")
    ax.set_yscale("log")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("Training Loss — EVE (ρ=0.58, K=4) vs SGD/AdamW/RMSProp", fontweight="bold")
plt.tight_layout()
plt.show()

# Validation accuracy curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, ds in zip(axes, DATASETS):
    for opt in OPTS:
        hist = results[ds][opt]["history"]
        ax.plot([h.epoch for h in hist], [h.val_acc for h in hist],
                label=opt, color=colors[opt])
    ax.axhline(TARGET_ACC[ds], color="gray", linestyle="--", alpha=0.7)
    ax.set_title(ds)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Val Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("Validation Accuracy — EVE (ρ=0.58, K=4) vs SGD/AdamW/RMSProp", fontweight="bold")
plt.tight_layout()
plt.show()

# Time to target bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(DATASETS) * len(OPTS))
width = 0.2
for i, opt in enumerate(OPTS):
    vals = [results[ds][opt]["time_to_target"] or 0 for ds in DATASETS]
    ax.bar(np.arange(len(DATASETS)) + i * width, vals, width, label=opt, color=colors[opt])
ax.set_xticks(np.arange(len(DATASETS)) + width * 1.5)
ax.set_xticklabels(DATASETS)
ax.set_ylabel("Seconds to target accuracy")
ax.set_title("Time to Converge (lower is better)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 9. Steps/sec (optimizer throughput) ───────────────────────────────────
print("Avg steps/sec per optimizer (higher = faster per-epoch):")
for ds in DATASETS:
    print(f"  {ds}: ", end="")
    for opt in OPTS:
        hist = results[ds][opt]["history"]
        sps = np.mean([h.steps_per_sec for h in hist])
        print(f"{opt}={sps:.0f} ", end="")
    print()